In [ ]:
# Include installs here in format !pip install xxxxx. Comment out if not needed for you
#!pip install mlflow
#!pip install kagglehub[pandas-datasets]

In [ ]:
# Include Includes here
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
import sys
import kagglehub
from sklearn.calibration import CalibratedClassifierCV
from kagglehub import KaggleDatasetAdapter
from sklearn.svm import SVC
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_curve, precision_recall_curve,
    confusion_matrix, auc
)
import mlflow
import mlflow.sklearn
from datetime import datetime
import logging

# Important
- Use MLFLOW for tracking

# 0. Problem Framing
We are using the dataset linked [here](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud). This dataset has a feature called `class` which contains the targets for our project. In this feature, an **anomaly** is defined as an instance of credit card fraud and is assigned a value of `1`. All non-anomalous transactions are assigned a value of `0`.

To ensure the best possible results given the information available, we will be using **supervised learning** with the `class` feature as our target variable. We will train three models, `Moving Average Bands`, `kNN`, and `SVM`. These models were selected specifically to compare the performance of statistical, distance-based, and machine learning approaches to anomaly detection. 

The contents of this dataset are wholly numeric results of PCA, except for `amount` and `time `. The original features are not provided to maintain privacy. The data is heavily imbalanced. As such, we will use primarily precision-recall curve for analysis of performance as the confusion matrix will not provide useful results. 
As per the dataset page, the non - PCA features are as follows:

> "Feature 'Time' contains the seconds elapsed between each transaction and the first transaction in the dataset. The feature 'Amount' is the transaction Amount, this feature can be used for example-dependant cost-sensitive learning."


## Method Discussion and Justification


## Moving Average Bands
-----
### Advantages
### Disadvantages
### Why it is good for this specific problem

## kNN
-----
### Advantages
### Disadvantages
### Why it is good for this specific problem


### SVM
-----
Support vector machines are an approach based on error-based learning. 
They are typically used for classification, regression, and outlier detection.
SVM is a **discriminative** classifier. This means it finds a point/line/plane/etc which divides classes from eachother.

If you draw a decision boundary or separating hyperplane between two classes, the distance from the boundary to the nearest training instance is known as the **margin**.
This can be illustrated using dashed lines known as **margin extents**. We optimize our model by selecting the separation with the greatest margin.

Where $\gamma_i$ is the distance between the boundary and a given point, we can write: 
$$
\gamma_i = y_i(w^Tx_i+b)\quad\quad y_i \in \{-1,1\}
$$

Here, $w$ is the separating hyperplane is given by
$$
b+w^T\cdot x=0
$$

Where $w\cdot d$ is the dot product of the weights and descriptive feature values, plus the bias (intercept).
for SVM, we first need to set the negative target feature level to -1 and the positive to +1. 
The model is then made such that instances with the negative target level output $\le-1$ and positive $\ge+1$.
The in-bewteen area allows for the margin.

Our goal is thus to find $b,w$ to maximize the margin, which can be achieved my minimizing $w$:

$$
\begin{array}{rl}
\textrm{max}_{\gamma,w,b} & \gamma \\
\textrm{s.t.} & y_i(w^Tx_i+b) \ge \gamma,\quad i=1...m\\
\end{array}
$$

$$
\begin{array}{rl}
\textrm{min}_{\gamma,w,b} & \frac{1}{2}||w||^2 \\
\textrm{s.t.} & y_i(w^Tx_i+b) \ge 1,\quad i=1...m\\
\end{array}
$$

A **hard margin** is that which was described above - and is useful in linearly seperable problems. 
We could also consider a **soft margin**, useful in situations where this is not the case or where the hard margin produces a small margin.
In this case, misclassification is allowed and then we minimize the error therein. 
The loss of a misclassified point is called a **slack variable**. 
We can then use the following equation for our problem.
$$
\begin{array}{rl}
\textrm{min}_{\gamma,w,b} & \frac{1}{2}||w||^2 + C \sum^m_{i=1}{\xi_i}\\
\textrm{s.t.} & y_i(w^Tx_i+b) \ge 1,\quad i=1...m\\
\textrm{    } & \xi_m \geq \forall m\\
\end{array}
$$


### Advantages
They are best for high-dimensional spaces and are memory efficient. 

When handling non-linearly separable datasets, we can change the **kernel**. 
This allows us to increase the dimensionality of our problem, helping to identify hyperplanes for classification.
By adjusting  $\gamma$, we can modify how far the influence of a training point is considered. 
The higher we set it, the closer points we consider.

We can vary the hyperparamter C to dictate the level of penalty applied to misclassification. 
A small C value implies no permission for error - or a hard boundary.
A large value of C implies a greater tolerance for misclassification - or a soft boundary.

### Disadvantages
If the number of features is far greater than that of samples, we need to employ regularization to avoid overfitting.
SVMs do not provide probability estimates, instead we need a five-fold cross validation to do this.

SVM is also very sensitive to feature scaling, and we should ensure all features are normalized or standardized.

### Why it is good for this specific problem
For anomaly detection, SVM will generate a decision function that assigns new points a decision score based on how anomalous they are.
By choosing the correct kernel and optimizing our hyperparameters - we are able to create a decision boundary should enclose the majority of the data.

Things to do for pretreating data
- check missing values
- oversample minority class
- split into train and test sets with stratified sampling using sklearn using ten fold cross validation so 10/90


## Data Understanding

# 1.Data Preparation

In [ ]:
def get_df():
    # make it so we dont need the local zip
    file_path = "creditcard.csv"
    df = kagglehub.load_dataset(
        KaggleDatasetAdapter.PANDAS,
        "mlg-ulb/creditcardfraud",
        file_path,
    )
    return df

In [ ]:
df = get_df()

# df = pd.read_csv("creditcard.csv")
rows = df.shape[0] #calculate number of rows
features = df.columns #features
n_features = df.shape[1] #number of features

print(f"Number of samples: {rows}")
print(f"Number of features: {n_features}")

print("\n\033[1m" + "Data types of each feature:" + "\033[0m")
for col in df.columns:
    print(f"\t{col:<10} = {df[col].dtype}")

#identify anomalies:

#find missing values in each feature:
if (df.isnull().any().any()): # If there are actually any missing values
    print(f"\nFeatures with missing values\n:")
    for feat in features:
        missing_nan = df[feat].isnull().sum() #adds all missing values in that column
        if missing_nan > 0:
            missing_per = (missing_nan/rows)*100 #finds the percentage
            print(f"{feat}: NaN = {missing_nan}, NaN% = {missing_per:.2f}%")
else:
    print("There are no missing values")

#find features with only one category (which provides no valuable information):
if (len(df.columns[df.nunique() == 1])>0): # If there are actually any
    print(f"\nFeatures with only one category:\n")
    for feat in features:
        if df[feat].nunique() == 1:
            print(f"{feat}")

df.head()

In [ ]:
# Correlation matrix
correlation_matrix = df.corr()

sns.heatmap(df.corr(), annot=False, cmap='coolwarm')
plt.title("Correlation Matrix Heatmap")
plt.show()

A low correlation among features generally implies that the variables are not strongly linearly related to each other. 
This also means that each feature is contributing different, non-redundant information.

To compensate for the severe class imbalance in the data, we will be using a **stratified 5-fold cross validation**. 
This will ensure that each training and validation split preserves the original class distribution.
This ensures that anomalies are represented in every fold, allowing for more reliable evaluation of the model’s detection performance
> MAYBE ADD MORE TO THIS?

## 1.1. Moving Average Bands

## 1.2. kNN

## 1.3. SVM

Robust scaling is used when outliers are real, but scaling is still necessary. 
In the case of supervised anomaly detection, we do not want to remove out anomalies. 
As such, we will use robust scaling - given by:
$$
x'=\frac{x-median(x)}{IQR(x)}
$$

In [ ]:
def SVM_preprocess(d):
    X = d.iloc[:, :-1]
    y = d.iloc[:, -1]

    scaler = RobustScaler()
    X_scaled = scaler.fit_transform(X)
    X_scaled = pd.DataFrame(X_scaled, columns=X.columns, index=X.index)

    return X_scaled,y

# 2. Detection Methods


## 2.1. Moving Average Bands

## 2.2. kNN

## 2.3. SVM

In [ ]:
def SVM_run(params, d, experiment_name):

    mlflow.set_experiment(experiment_name)
    

    run_name = f"SVM_k={params['kernel']}_C={params['C']:.2g}_t={params['threshold']:.2g}"
    print(f"\n{run_name}")

    with mlflow.start_run(run_name=run_name):

        # Preprocess
        X, y = SVM_preprocess(d)
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        print("Preprocessing Success")

        # Log params
        mlflow.log_params(params)
        print("Param Log Success")

        # Build model
        model = SVC(
            kernel=params["kernel"],
            C=params["C"],
            degree=params.get("degree", 3),
            gamma=params.get("gamma", "scale"),
            probability=True,
            random_state=42
        )
        print("Model Build Success")
        

        # Predictions - Slow step 
        probs = cross_val_predict(model, X, y, cv=skf, method="predict_proba",n_jobs=-1)
        y_proba = probs[:, 1]
        y_pred = (y_proba >= params['threshold']).astype(int)
        print("Predict Success")

        # Metrics
        precision = precision_score(y, y_pred)
        recall = recall_score(y, y_pred)
        f1 = f1_score(y, y_pred)
        print("Metric Calc Success")

        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)
        print("Metric Log Success")

        # Confusion matrix + FPR
        cm = confusion_matrix(y, y_pred)
        tn, fp, fn, tp = cm.ravel()
        fpr_value = fp / (fp + tn)
        print("CM Calc Success")

        mlflow.log_metric("false_positive_rate", fpr_value)
        print("FPR Log Success")

        # ROC
        fpr, tpr, _ = roc_curve(y, y_proba)
        roc_auc = auc(fpr, tpr)
        print("ROC/AUC Calc Success")

        mlflow.log_metric("roc_auc", roc_auc)
        print("ROC/AUC Log Success")

        # PR curve
        precision_curve, recall_curve, _ = precision_recall_curve(y, y_proba)
        print("PR Curve Calc Success")

        # Fit + log model 
        model.fit(X, y)
        mlflow.sklearn.log_model(model, name="svm_model")
        print("Model Log Success")

        # Tags
        mlflow.set_tag("model", "SVM")
        mlflow.set_tag("kernel", params["kernel"])
        mlflow.set_tag("threshold", params["threshold"])

    return cm, fpr, tpr, roc_auc, precision_curve, recall_curve

# 3. Threshold Optimization
- Optimizing to reduce false negatives
- Confusion matrix less important here
- AUC more important
- Analyze anomaly score distributions
- Select and justify detection threshold
- Perform sensitivity analysis on threshold selection
- Priority = recall

## 3.1. Moving Average Bands

## 3.2. kNN

## 3.3. SVM

In [ ]:
C_values = np.logspace(-5, 0, 15)
gamma_values = np.logspace(-6, 1, 15)
thresholds = np.arange(0.0, 1.01, 0.05)

kernels = ["linear", "rbf","poly", "sigmoid"]
degrees = [2, 3, 4, 5]

param_grid = []

for kernel in kernels:
    for C in C_values:
        for t in thresholds:
            if kernel == "linear":
                param_grid.append({
                    "kernel": kernel,
                    "C": float(C),
                    "threshold": t
                })

            elif kernel == "poly":
                for gamma in gamma_values:
                    for degree in degrees:
                        param_grid.append({
                            "kernel": kernel,
                            "C": float(C),
                            "gamma": float(gamma),
                            "degree": degree,
                            "threshold": t
                        })

            else:  # rbf, sigmoid
                for gamma in gamma_values:
                    param_grid.append({
                        "kernel": kernel,
                        "C": float(C),
                        "gamma": float(gamma),
                        "threshold": t
                    })


df = get_df()

# Function wrapper for try-except
def run_svm(params):
    try:
        SVM_run(params, df, "SVM_v1_colab")
    except Exception as e:
        return f"ERROR with {params}: {e}"
    return f"SUCCESS with {params}"

# run
max_workers = 15 
with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = [executor.submit(run_svm, p) for p in param_grid]
    
    for future in as_completed(futures):
        result = future.result()
        if result.startswith("ERROR"):
            print(result)

# 4. Evaluation
- Precision, Recall, F1-score
- ROC curve and PR curve
- Confusion matrix at selected threshold
- False positive rate analysis

## 4.1. MAB

## 4.2. kNN

## 4.3. SVM

# 5. Interperabiltiy

# Reflection
- Trade-offs in threshold selection
- Impact of imbalance
- Failure scenarios in production
- Monitoring and retraining strategy
- Logged experiments
- Discussion of production risks and monitoring